# Imputación usando MissForest

In [ ]:
# ################################################################################### #
# Script used for the imputation of the missing values of the dataset by means of the #
# technique MissForest. Reference:                                                    #
#                                                                                     #
#           Daniel J. Stekhoven and Peter Bühlmann. MissForest - nonparametric        #
#           missing value imputation for mixed-type data. 2011, 28, 1, 112-118.       #
#           DOI: 10.1093/bioinformatics/btr597.                                       #
#                                                                                     #
# ################################################################################### #    

## Idea general


1. Imputar numéricas + categóricas ordinales con “MissForest” (IterativeImputer + árboles)

2. Para categóricas nominales, NO imputarlas como números ni como one-hot continuo; imputarlas como categoría (más simple y estable).

Es necesario hacer esta separación porque:

- **Las ordinales sí tienen orden** → tiene sentido imputarlas como números y luego redondear

- **Las nominales no tienen orden** → imputarlas como números mete orden falso; imputarlas como one-hot con modelos de regresión deja dummies inconsistentes.

In [1]:
# Import basics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import joblib
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)

In [ ]:
df = pd.read_csv('', delimiter=';')
df

In [ ]:
# Para asegurar que el imputador funcione bien convertimos los valores faltantes, representados en blanco, a NaN
df.replace("", np.nan, inplace=True)
df

In [ ]:
df.shape

In [ ]:
df.columns

### Esta es la celda que usaríamos si antes hiciesemos split en train/test

In [14]:
# Funciones para la imputación
def _missForest_fit(data, num_estimators, m_depth, m_samples_leaf, max_iter,
                    min_value=None, max_value=None, random_state=1234):
    tree = ExtraTreesRegressor(n_estimators=num_estimators,
                               random_state=random_state,
                               max_depth=m_depth,
                               min_samples_leaf=m_samples_leaf)
    imputer = IterativeImputer(estimator=tree,
                               random_state=random_state,
                               max_iter=max_iter,
                               min_value=min_value,
                               max_value=max_value,
                               imputation_order='ascending',
                               initial_strategy='mean',
                               verbose=2)
    imputer.fit(data)
    return imputer

def _missForest_transform(imputer, data):
    return imputer.transform(data)


def imputacion_pipeline(train_num, test_num, train_cat, test_cat,
                        num_estimators, m_depth, m_samples_leaf,
                        max_iter, random_state=1234,
                        guardar_imputador=True):

    # Obtener min/max automáticamente por nombre de variable
    min_values_num = [min_values_dict[col] for col in train_num.columns]
    max_values_num = [max_values_dict[col] for col in train_num.columns]

    min_values_all = [min_values_dict[col] for col in list(train_num.columns) + list(train_cat.columns)]
    max_values_all = [max_values_dict[col] for col in list(train_num.columns) + list(train_cat.columns)]

    
    # Imputación de numéricas
    imputer_num = _missForest_fit(train_num, num_estimators, m_depth, m_samples_leaf,
                                  max_iter, min_value=min_values_num,
                                  max_value=max_values_num,
                                  random_state=random_state)
    train_num_imp = pd.DataFrame(_missForest_transform(imputer_num, train_num),
                                 columns=train_num.columns,
                                 index=train_num.index)
    test_num_imp = pd.DataFrame(_missForest_transform(imputer_num, test_num),
                                columns=test_num.columns,
                                index=test_num.index)

    # Unir con categóricas
    train_all = pd.concat([train_num_imp, train_cat], axis=1)
    test_all = pd.concat([test_num_imp, test_cat], axis=1)

    # Imputación conjunta
    imputer_all = _missForest_fit(train_all, num_estimators, m_depth, m_samples_leaf,
                                  max_iter, min_value=min_values_all,
                                  max_value=max_values_all,
                                  random_state=random_state)
    train_all_imp = _missForest_transform(imputer_all, train_all)
    test_all_imp = _missForest_transform(imputer_all, test_all)

    # Control de variables categóricas
    for i in range(train_num.shape[1], train_all.shape[1]):
        train_all_imp[:, i] = np.clip(train_all_imp[:, i], min_values_all[i], max_values_all[i])
        test_all_imp[:, i] = np.clip(test_all_imp[:, i], min_values_all[i], max_values_all[i])

    train_final = pd.DataFrame(train_all_imp, columns=train_all.columns, index=train_all.index)
    test_final = pd.DataFrame(test_all_imp, columns=test_all.columns, index=test_all.index)

    for col in train_cat.columns:
        train_final[col] = train_final[col].round().astype(int)
        test_final[col] = test_final[col].round().astype(int)

    if guardar_imputador:
        joblib.dump(imputer_all, 'imputador_missforest.pkl')
        print("Imputador guardado como 'imputador_missforest.pkl'")

    return train_final, test_final

## Flujo general del código

**1. Definición de rangos permitidos (min_value y max_value)**

El imputador MissForest, a través de IterativeImputer, puede imputar valores fuera de los rangos clínicos reales si no se le ponen restricciones.  Por ello, se definen dos diccionarios: min_values_dict (mapeo de cada variable a su valor mínimo aceptado) y max_values_dict (mapeo de cada variable, su valor máximo aceptado). Esto se hace por nombre de variable para evitar errores si se cambia el orden de las columnas.

**2. Entrenamiento del imputador MissForest solo en datos de entrenamiento**

Este paso es necesario para evitar data leakage, el imputador se entrena solo con datos de entrenamiento. Luego se aplica al conjunto de test sin volver a ajustar.

**3. Aplicación del imputador a test**

El conjunto de test debe representar datos nunca vistos. Solo podemos usar el imputador previamente entrenado en train, ya que en producción no conoceremos la distribución real del test.

**4. Unión de variables categóricas (no imputadas todavía)**

Las categóricas también deben imputarse si tienen valores faltantes. Por eficiencia y consistencia, se hace una segunda imputación sobre todo el conjunto combinado (números ya imputados + categóricas originales).

**5. Imputación conjunta de numéricas + categóricas**

Esta segunda imputación considera interacciones entre variables numéricas y categóricas. Se usan min_values_all y max_values_all que incluyen las restricciones de todas las variables.

**6. Clipping y redondeo de variables categóricas**

Aunque se hayan definido rangos, MissForest puede producir pequeños decimales. Las variables categóricas deben ser redondeadas al entero más cercano y convertidas a int.

**7. Guardado del imputador para producción**

Guardar el imputador permite reutilizarlo en producción (por ejemplo, en una API Flask), garantizando que los datos futuros se traten exactamente igual que los del entrenamiento.

### Esta es la celda que utilizamos sin dividir en train/test 

In [ ]:
# Funciones para la imputación
def _missForest_fit(data, num_estimators, m_depth, m_samples_leaf, max_iter,
                    min_value=None, max_value=None, random_state=1234):
    tree = ExtraTreesRegressor(
        n_estimators=num_estimators,
        random_state=random_state,
        max_depth=m_depth,
        min_samples_leaf=m_samples_leaf
    )
    imputer = IterativeImputer(
        estimator=tree,
        random_state=random_state,
        max_iter=max_iter,
        min_value=min_value,
        max_value=max_value,
        imputation_order='ascending',
        initial_strategy='mean',
        verbose=2
    )
    imputer.fit(data)
    return imputer

def _missForest_transform(imputer, data):
    return imputer.transform(data)


def imputacion_pipeline_all(data_num, data_cat,
                            num_estimators, m_depth, m_samples_leaf,
                            max_iter, random_state=1234,
                            guardar_imputador=True):

    # Obtener min/max automáticamente por nombre de variable
    min_values_num = [min_values_dict[col] for col in data_num.columns]
    max_values_num = [max_values_dict[col] for col in data_num.columns]

    min_values_all = [min_values_dict[col] for col in list(data_num.columns) + list(data_cat.columns)]
    max_values_all = [max_values_dict[col] for col in list(data_num.columns) + list(data_cat.columns)]

    # Imputación de numéricas
    imputer_num = _missForest_fit(
        data_num, num_estimators, m_depth, m_samples_leaf,
        max_iter, min_value=min_values_num,
        max_value=max_values_num,
        random_state=random_state
    )

    data_num_imp = pd.DataFrame(
        _missForest_transform(imputer_num, data_num),
        columns=data_num.columns,
        index=data_num.index
    )

    # Unir con categóricas (ordinales)
    data_all = pd.concat([data_num_imp, data_cat], axis=1)

    # Imputación conjunta
    imputer_all = _missForest_fit(
        data_all, num_estimators, m_depth, m_samples_leaf,
        max_iter, min_value=min_values_all,
        max_value=max_values_all,
        random_state=random_state
    )

    data_all_imp = _missForest_transform(imputer_all, data_all)

    # Control de variables categóricas (ordinales)
    for i in range(data_num.shape[1], data_all.shape[1]):
        data_all_imp[:, i] = np.clip(data_all_imp[:, i], min_values_all[i], max_values_all[i])

    data_final = pd.DataFrame(data_all_imp, columns=data_all.columns, index=data_all.index)

    for col in data_cat.columns:
        data_final[col] = data_final[col].round().astype(int)

    if guardar_imputador:
        joblib.dump(imputer_all, 'imputador_missforest.pkl')
        print("Imputador guardado como 'imputador_missforest.pkl'")

    return data_final

## Los rangos min_value y max_value deben calcularse exclusivamente sobre el conjunto de train

Así evitamos **data leakage**. Si extraemos los valores mínimos y máximos de todo el dataset, estarías indirectamente "mirando" los datos de test durante el entrenamiento del imputador → esto contamina el modelo.

**Coherencia con la imputación**: MissForest usa los rangos para limitar las predicciones durante el proceso iterativo. Si esos rangos provienen de test, estás condicionando la forma en la que el modelo "aprende" los valores del train.

**Producción realista**: En un entorno real, nunca dispones del test en el momento de entrenar el modelo. Así que los rangos utilizados deben derivarse únicamente de los datos conocidos (train).

**¿Qué pasa si en test hay valores fuera de los rangos de train?**

Esto no debería ocurrir si: hemos realizado una buena división estratificada y la muestra de train es representativa.

Pero si ocurre, no debemos ajustar los rangos. En su lugar, podemos:

- Aplicar un clip() posterior para limitar valores extremos en test.

- Analizar si el test contiene poblaciones diferentes y considerar un rediseño del muestreo.

In [15]:
min_values_dict = df.min().to_dict()
max_values_dict = df.max().to_dict()

In [ ]:
min_values_dict

In [ ]:
max_values_dict

## ¿Por qué se realiza la imputación en 2 pasos: primero variables numéricas y luego numéricas+categóricas?

Imputamos primero solo las variables numéricas por:

1. Mejor estabilidad del imputador numérico

Los algoritmos como MissForest (basado aquí en IterativeImputer + ExtraTreesRegressor) pueden beneficiarse de entrenarse primero solo con variables numéricas. Esto:

- Reduce el ruido introducido por las categóricas codificadas como números

- Mejora la estimación de patrones numéricos y relaciones entre variables continuas.

- Permite detectar y excluir variables problemáticas antes de mezclar tipos de datos (por ejemplo, variables constantes o con distribución extraña).

2. Control independiente de los rangos

A nivel práctico, se puede definir una lista más sencilla de min_values y max_values solo para variables numéricas y verificar su comportamiento antes de introducir restricciones adicionales para categóricas.

**¿Por qué luego se imputan conjuntamente las numéricas (ya imputadas) y las categóricas?**

Una vez imputadas las numéricas, se hace una segunda imputación conjunta de: numéricas_imputadas + categóricas_con_missing


1. Mejora de la imputación de categóricas

Las variables categóricas se imputan mejor si se aprovecha la información de las variables numéricas completas. Por ejemplo:

La paridad puede estar relacionada con la edad materna o con el IMC.

La codificación de Doppler puede depender del flujo sanguíneo cuantificado en variables continuas.

2. Coherencia entre variables

Permite capturar dependencias cruzadas entre tipos de variables (ej. una categoría que suele coincidir con valores altos en una variable continua).

3. No se reimputan las numéricas

En esta segunda etapa, las numéricas ya imputadas no se modifican, pero participan como predictores para imputar las categóricas. (El IterativeImputer respeta valores ya imputados si se le pasan como entrada sin NaNs).

In [19]:
# Diferenciar variables numéricas y categóricas nominales y ordinales
num_cols = ['', '', ]
cat_cols_ord = ['', '', ]
cat_cols_nom = ['', '', ]
# División
df_num = df[num_cols]
df_cat_ord = df[cat_cols_ord]
df_cat_nom = df[cat_cols_nom]

Los rangos tienen que estar ordenados en función de lo que "entre" a la imputación.

1. Imputación de las variables numéricas (train_num): rangos min/max en este orden

2. Imputación de las variables numéricas + categóricas: aquí lo que "entra" a la imputación es train_all = pd.concat([train_num, train_cat], axis=1). Por lo tanto, el orden correcto de los rangos min/max será primero todas las numéricas y luego todas las categóricas. 

In [ ]:
df_num

In [ ]:
# Probar esto para ver que los min/max son correctos
min_values_num = [min_values_dict[col] for col in df_num.columns]
max_values_num = [max_values_dict[col] for col in df_num.columns]
min_values_num

In [ ]:
min_values_all = [min_values_dict[col] for col in list(df_num.columns) + list(df_cat_ord.columns)]
max_values_all = [max_values_dict[col] for col in list(df_num.columns) + list(df_cat_ord.columns)]
max_values_all

### Importante 

Aunque MissForest solo modifica las variables con valores faltantes, el imputador necesita todas las columnas para predecir, por eso hay que establecer correctamente los rangos min_value y max_value para todas las columnas, incluso las que no tienen valores faltantes, para evitar imputaciones fuera de rango cuando esas columnas sirven como predictores.

### Función cuando hacemos split en train/test

In [ ]:
# Llamar a la función para imputar
train_final, test_final = imputacion_pipeline(
    train_num=train_num,
    test_num=test_num,
    train_cat=train_cat,
    test_cat=test_cat,
    num_estimators=50, # Número de árboles en el modelo ExtraTrees
    m_depth=4, # Profundidad máxima de los árboles
    m_samples_leaf=3, # Mínimo de muestras en cada hoja
    max_iter=100,
    random_state=1234,
    guardar_imputador=True
)

### Función sin split en train/test

In [ ]:
data_final = imputacion_pipeline_all(
    data_num=df_num,
    data_cat=df_cat_ord,   # aquí SOLO ordinales (codificadas como enteros)
    num_estimators=50,
    m_depth=4,
    m_samples_leaf=3,
    max_iter=100,
    random_state=1234,
    guardar_imputador=True)

### Imputación de las variables categóricas nominales usando la moda

In [ ]:
def impute_nominal_simple(X_nom):
    imp = SimpleImputer(strategy='most_frequent')
    X_nom_imp = pd.DataFrame(imp.fit_transform(X_nom), columns=X_nom.columns, index=X_nom.index)
    return X_nom_imp, imp

In [ ]:
X_nom_imp, imp = impute_nominal_simple(df_cat_nom)

### Unión de todas las variables

In [ ]:
df_imp = pd.concat([data_final, X_nom_imp], axis=1)

In [ ]:
# Comprobar que no hay valores faltantes
df_imp.isna().sum().sum()

In [ ]:
df_imp

## Comparación de la distribución

En esta parte lo que comparamos es:

- **Distribución original**: los valores no nulos originales 

- **Distribución imputada completa**: todos los valores del train tras imputar, que incluyen imputados + originales.

Así podemos responder preguntas como: **¿La imputación ha distorsionado de forma visible la forma de la distribución?**

In [ ]:
# Estadísticas antes de la imputación
df.describe()

In [ ]:
df_imp.describe()

### A partir de aquí ya vendría:

- One-Hot para categóricas nominales
- Reagrupación de categorías poco frecuentes
- Transformación de variables
- Creación de variables derivadas
- Normalización/estandarización de variables numéricas